# 03 — Evaluation

Runs the evaluation suite on CPU and displays the JSON reports:

- `src.evaluation.evaluate` — answer quality on the test split, including the
  competition-style **Levenshtein Mean** metric
- `src.evaluation.evaluate_rag` — retrieval quality (top-k hit rate over the TF-IDF index)
- `src.evaluation.benchmark` — latency / throughput of the inference pipeline

Pure-Python fallbacks in `src/evaluation/metrics.py` mean everything here runs without a GPU
(optional `rouge-score` / `nltk` / `scikit-learn` enrich the reports when installed).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists() and (ROOT.parent / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

## Run the three evaluation modules

Each module exposes a `main()` entry point; the returned dicts are printed as JSON.

In [ ]:
import json

from src.evaluation import evaluate, evaluate_rag, benchmark

reports = {}
for mod in (evaluate, evaluate_rag, benchmark):
    entry = getattr(mod, "main", None)
    print("=" * 60)
    print(mod.__name__)
    if entry is None:
        print("no main() entry point found")
        continue
    out = entry()
    reports[mod.__name__] = out
    print(json.dumps(out, indent=2, default=str)[:4000])

## Saved JSON reports

If the evaluation modules persisted reports to disk, list and preview them here.

In [ ]:
import json

for report in sorted(ROOT.glob("reports/*.json")):
    print("=" * 60)
    print(report.name)
    data = json.loads(report.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2, default=str)[:3000])
else:
    print("(no reports/*.json files found — reports were shown inline above)")